<a href="https://colab.research.google.com/github/mandevautospa/GlobalTerror_NeuralNetwork/blob/main/gtd_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pandas as pd
import kagglehub
import torch
import torch.nn as nn
import requests
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics.pairwise import haversine_distances

path = kagglehub.dataset_download("START-UMD/gtd")
print(tf.version) #ensure version 2.x

Using Colab cache for faster access to the 'gtd' dataset.
<module 'tensorflow._api.v2.version' from '/usr/local/lib/python3.12/dist-packages/tensorflow/_api/v2/version/__init__.py'>


In [ ]:
for f in os.listdir(path):
  if f.endswith(".csv"):
    csv_path = os.path.join(path, f)
df = pd.read_csv(csv_path, encoding ="latin1", low_memory=False)
df = df.replace({-99:None})
df.head()

features = ["iyear", "imonth", "iday",
            "country", "region",
            "latitude", "longitude",
            "nkill", "nwound", "gname"]

targets = ["attacktype1", "success", "gname", "nkill", "nwound"]
all_cols = list(dict.fromkeys(features + targets))
df = df[all_cols].dropna()

encoders = {}
cat_cols = ["country", "region", "gname", "attacktype1"]

for col in cat_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col].astype(str))
  encoders[col] = le

# All numeric feature columns (scaling deferred to after train/val split)
num_cols = ["iyear", "imonth", "iday", "latitude", "longitude", "nkill", "nwound"]


In [ ]:
import os

# iso3_map: map ACLED country names to ISO 3166-1 alpha-3 codes
iso3_map = {}

ACLED_URL = "https://api.acleddata.com/acled/read"

# Retrieve ACLED credentials from Colab userdata or environment variables
try:
    from google.colab import userdata
    acled_email = userdata.get("ACLED_EMAIL")
    acled_key   = userdata.get("ACLED_KEY")
except Exception:
    acled_email = os.environ.get("ACLED_EMAIL", "")
    acled_key   = os.environ.get("ACLED_KEY", "")

params = {"limit": 1000, "year": 2026, "email": acled_email, "key": acled_key}

try:
    r = requests.get(ACLED_URL, params=params, timeout=30)
    r.raise_for_status()
    acled = pd.DataFrame(r.json()['data'])
    acled['event_date_utc'] = pd.to_datetime(acled['event_date'], utc=True)
    acled['country_iso3'] = acled['country'].map(iso3_map)
    acled['reported_by_acled'] = 1
except Exception as e:
    print(f"ACLED data unavailable: {e}")
    acled = pd.DataFrame(columns=['event_date', 'event_date_utc', 'country', 'country_iso3', 'reported_by_acled'])

In [ ]:
print(df.columns.tolist())

['iyear', 'imonth', 'iday', 'country', 'region', 'latitude', 'longitude', 'nkill', 'nwound', 'gname', 'attacktype1', 'success']


In [ ]:
# Temporal train/validation split: train on older data, validate on newer data
# Scaler is fit on training data only to prevent data leakage
# split_year can be adjusted: earlier years give more training data,
# later years give a larger/more-recent validation set
split_year = 2015
df_train = df[df["iyear"] <= split_year].copy()
df_val   = df[df["iyear"] > split_year].copy()

scaler = StandardScaler()
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
df_val[num_cols]   = scaler.transform(df_val[num_cols])


In [ ]:
X_train = torch.tensor(df_train.drop(columns=targets).values, dtype=torch.float32)
X_val   = torch.tensor(df_val.drop(columns=targets).values, dtype=torch.float32)

y_attacktype_train = torch.tensor(df_train["attacktype1"].values, dtype=torch.long)
y_success_train    = torch.tensor(df_train["success"].values, dtype=torch.float32)
y_group_train      = torch.tensor(df_train["gname"].values, dtype=torch.long)
y_casualties_train = torch.tensor(df_train[["nkill", "nwound"]].values, dtype=torch.float32)

y_attacktype_val   = torch.tensor(df_val["attacktype1"].values, dtype=torch.long)
y_success_val      = torch.tensor(df_val["success"].values, dtype=torch.float32)
y_group_val        = torch.tensor(df_val["gname"].values, dtype=torch.long)
y_casualties_val   = torch.tensor(df_val[["nkill", "nwound"]].values, dtype=torch.float32)


In [ ]:
# ── 1. Unified multi-output TensorDataset & DataLoaders ───────────────────
train_dataset = TensorDataset(
    X_train,
    y_attacktype_train,
    y_success_train,
    y_group_train,
    y_casualties_train
)

val_dataset = TensorDataset(
    X_val,
    y_attacktype_val,
    y_success_val,
    y_group_val,
    y_casualties_val
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


In [ ]:
# ── 2. Multi-task neural network ─────────────────────────────────────────
class MultiTaskGTD(nn.Module):
    """Shared encoder with four task-specific output heads.

    Heads
    -----
    attack_type : multi-class  (CrossEntropyLoss)
    success     : binary       (BCEWithLogitsLoss)
    group       : multi-class  (CrossEntropyLoss)
    casualties  : 2-dim regression for [nkill, nwound] (MSELoss)
    """

    def __init__(self, input_dim, num_attacktypes, num_groups, hidden_dim=256, dropout=0.3):
        super().__init__()

        # Shared encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Task heads
        self.head_attacktype = nn.Linear(hidden_dim, num_attacktypes)
        self.head_success    = nn.Linear(hidden_dim, 1)
        self.head_group      = nn.Linear(hidden_dim, num_groups)
        self.head_casualties = nn.Linear(hidden_dim, 2)  # outputs [nkill, nwound]

    def forward(self, x):
        shared = self.encoder(x)
        attacktype_out = self.head_attacktype(shared)
        success_out    = self.head_success(shared).squeeze(1)  # (B,)
        group_out      = self.head_group(shared)
        casualties_out = self.head_casualties(shared)
        return attacktype_out, success_out, group_out, casualties_out


# Infer dimensions from tensors
# Use torch.unique to handle non-contiguous or missing label values safely
input_dim       = X_train.shape[1]
num_attacktypes = len(torch.unique(y_attacktype_train))
num_groups      = len(torch.unique(y_group_train))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = MultiTaskGTD(input_dim, num_attacktypes, num_groups).to(device)
print(model)
print(f"\ninput_dim={input_dim}, num_attacktypes={num_attacktypes}, num_groups={num_groups}")


In [ ]:
# ── Class distribution analysis ─────────────────────────────────────────
# Inspect normalized class frequencies before training to detect imbalance.

def plot_class_distribution(df, encoders):
    """Print normalized value counts for attacktype1 and top-20 gname groups."""
    # attacktype1 distribution
    att_counts = df["attacktype1"].value_counts(normalize=True)
    att_labels = encoders["attacktype1"].inverse_transform(att_counts.index.astype(int))
    print("=== attacktype1 class distribution (normalized) ===")
    for label, freq in zip(att_labels, att_counts.values):
        print(f"  {label:40s}: {freq:.4f}")

    # gname distribution (top 20 only)
    gname_counts = df["gname"].value_counts(normalize=True).head(20)
    gname_labels = encoders["gname"].inverse_transform(gname_counts.index.astype(int))
    print("\n=== gname class distribution – top 20 groups (normalized) ===")
    for label, freq in zip(gname_labels, gname_counts.values):
        print(f"  {label:40s}: {freq:.4f}")

    # Comment on severity of imbalance
    max_att = att_counts.values[0]
    print(f"\nattacktype1 most-frequent class share: {max_att:.2%}")
    if max_att > 0.5:
        print("  → Severe class imbalance detected for attacktype1.")
    else:
        print("  → Class distribution for attacktype1 is relatively balanced.")

    max_gname = gname_counts.values[0]
    print(f"gname most-frequent group share (top-20): {max_gname:.2%}")
    if max_gname > 0.5:
        print("  → Severe class imbalance detected for gname.")
    else:
        print("  → Class distribution for gname is relatively balanced.")


plot_class_distribution(df, encoders)


In [ ]:
# ── 3 & 4. Multi-task training + validation loop ─────────────────────────
BINARY_THRESHOLD = 0.0  # decision boundary for success head logits

ce_loss  = nn.CrossEntropyLoss()
bce_loss = nn.BCEWithLogitsLoss()
mse_loss = nn.MSELoss()
mae_loss = nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 20

# Optional loss weighting – adjust to up/downweight specific tasks
loss_weights = {
    "attack":     1.0,
    "success":    1.0,
    "group":      1.0,
    "casualties": 0.5,
}

# History dictionary – one list per metric, appended each epoch
history = {
    "train_loss": [], "val_loss": [],
    "train_att_loss": [], "val_att_loss": [],
    "train_suc_loss": [], "val_suc_loss": [],
    "train_grp_loss": [], "val_grp_loss": [],
    "train_cas_loss": [], "val_cas_loss": [],
    "val_att_acc": [], "val_suc_acc": [], "val_grp_acc": [],
    "val_cas_mse": [], "val_cas_mae": [],
}

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    total_loss = total_att = total_suc = total_grp = total_cas = 0.0

    for X_b, y_att, y_suc, y_grp, y_cas in train_loader:
        X_b   = X_b.to(device)
        y_att = y_att.to(device)
        y_suc = y_suc.to(device)
        y_grp = y_grp.to(device)
        y_cas = y_cas.to(device)

        att_out, suc_out, grp_out, cas_out = model(X_b)

        loss_attack     = ce_loss(att_out, y_att)
        loss_success    = bce_loss(suc_out, y_suc.float())
        loss_group      = ce_loss(grp_out, y_grp)
        loss_casualties = mse_loss(cas_out, y_cas.float())

        loss = (loss_weights["attack"]     * loss_attack
              + loss_weights["success"]    * loss_success
              + loss_weights["group"]      * loss_group
              + loss_weights["casualties"] * loss_casualties)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_att  += loss_attack.item()
        total_suc  += loss_success.item()
        total_grp  += loss_group.item()
        total_cas  += loss_casualties.item()

    n = len(train_loader)
    train_log = (
        f"Epoch {epoch:>2}/{NUM_EPOCHS} | "
        f"Train total={total_loss/n:.4f}  "
        f"att={total_att/n:.4f}  "
        f"suc={total_suc/n:.4f}  "
        f"grp={total_grp/n:.4f}  "
        f"cas={total_cas/n:.4f}"
    )

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    v_loss = v_att = v_suc = v_grp = v_cas = v_cas_mae = 0.0
    correct_att = correct_suc = correct_grp = 0
    total_samples = 0

    with torch.no_grad():
        for X_b, y_att, y_suc, y_grp, y_cas in val_loader:
            X_b   = X_b.to(device)
            y_att = y_att.to(device)
            y_suc = y_suc.to(device)
            y_grp = y_grp.to(device)
            y_cas = y_cas.to(device)

            att_out, suc_out, grp_out, cas_out = model(X_b)

            v_att     += ce_loss(att_out, y_att).item()
            v_suc     += bce_loss(suc_out, y_suc.float()).item()
            v_grp     += ce_loss(grp_out, y_grp).item()
            v_cas     += mse_loss(cas_out, y_cas.float()).item()
            v_cas_mae += mae_loss(cas_out, y_cas.float()).item()

            batch_size = X_b.size(0)
            total_samples += batch_size

            correct_att += (att_out.argmax(1) == y_att).sum().item()
            correct_suc += ((suc_out > BINARY_THRESHOLD).long() == y_suc.long()).sum().item()
            correct_grp += (grp_out.argmax(1) == y_grp).sum().item()

    m = len(val_loader)
    acc_att = correct_att / total_samples
    acc_suc = correct_suc / total_samples
    acc_grp = correct_grp / total_samples
    val_total = v_att + v_suc + v_grp + v_cas

    val_log = (
        f"          | "
        f"  Val total={val_total/m:.4f}  "
        f"att={v_att/m:.4f}(acc={acc_att:.3f})  "
        f"suc={v_suc/m:.4f}(acc={acc_suc:.3f})  "
        f"grp={v_grp/m:.4f}(acc={acc_grp:.3f})  "
        f"cas_mse={v_cas/m:.4f}"
    )

    print(train_log)
    print(val_log)

    # ── Append to history ─────────────────────────────────────────────────
    history["train_loss"].append(total_loss / n)
    history["val_loss"].append(val_total / m)
    history["train_att_loss"].append(total_att / n)
    history["val_att_loss"].append(v_att / m)
    history["train_suc_loss"].append(total_suc / n)
    history["val_suc_loss"].append(v_suc / m)
    history["train_grp_loss"].append(total_grp / n)
    history["val_grp_loss"].append(v_grp / m)
    history["train_cas_loss"].append(total_cas / n)
    history["val_cas_loss"].append(v_cas / m)
    history["val_att_acc"].append(acc_att)
    history["val_suc_acc"].append(acc_suc)
    history["val_grp_acc"].append(acc_grp)
    history["val_cas_mse"].append(v_cas / m)
    history["val_cas_mae"].append(v_cas_mae / m)


In [ ]:
import matplotlib
import matplotlib.pyplot as plt


def plot_learning_curves(history):
    """Plot total loss, per-task losses, accuracy curves, and regression metrics."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Training Diagnostics", fontsize=14)

    # Total loss
    ax = axes[0, 0]
    ax.plot(epochs, history["train_loss"], label="Train Total Loss")
    ax.plot(epochs, history["val_loss"],   label="Val Total Loss")
    ax.set_title("Total Loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True)

    # Per-task losses
    ax = axes[0, 1]
    for task, color in [("att", "blue"), ("suc", "orange"), ("grp", "green"), ("cas", "red")]:
        ax.plot(epochs, history[f"train_{task}_loss"], linestyle="--", color=color,
                alpha=0.6, label=f"Train {task}")
        ax.plot(epochs, history[f"val_{task}_loss"],   linestyle="-",  color=color,
                label=f"Val {task}")
    ax.set_title("Per-Task Losses")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(fontsize=7); ax.grid(True)

    # Per-task accuracy (classification tasks)
    ax = axes[1, 0]
    ax.plot(epochs, history["val_att_acc"], label="attacktype1 Accuracy")
    ax.plot(epochs, history["val_suc_acc"], label="success Accuracy")
    ax.plot(epochs, history["val_grp_acc"], label="gname Accuracy")
    ax.set_title("Validation Accuracy (Classification)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(); ax.grid(True)

    # Regression metrics (casualties MSE / MAE)
    ax = axes[1, 1]
    ax.plot(epochs, history["val_cas_mse"], label="Casualties MSE")
    ax.plot(epochs, history["val_cas_mae"], label="Casualties MAE")
    ax.set_title("Casualty Regression Metrics (Validation)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Error")
    ax.legend(); ax.grid(True)

    plt.tight_layout()
    plt.show()


plot_learning_curves(history)


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns


def plot_confusion_matrices(model, val_loader, encoders, device, top_n_groups=10):
    """Plot confusion matrices for attacktype1 and top-N gname groups."""
    model.eval()
    all_att_true, all_att_pred = [], []
    all_grp_true, all_grp_pred = [], []

    with torch.no_grad():
        for X_b, y_att, y_suc, y_grp, y_cas in val_loader:
            X_b = X_b.to(device)
            att_out, _, grp_out, _ = model(X_b)
            all_att_true.extend(y_att.cpu().numpy())
            all_att_pred.extend(att_out.argmax(1).cpu().numpy())
            all_grp_true.extend(y_grp.cpu().numpy())
            all_grp_pred.extend(grp_out.argmax(1).cpu().numpy())

    all_att_true = np.array(all_att_true)
    all_att_pred = np.array(all_att_pred)
    all_grp_true = np.array(all_grp_true)
    all_grp_pred = np.array(all_grp_pred)

    # ── attacktype1 confusion matrix ──────────────────────────────────────
    att_classes = encoders["attacktype1"].classes_
    cm_att = confusion_matrix(all_att_true, all_att_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_att, annot=True, fmt="d", cmap="Blues",
                xticklabels=att_classes, yticklabels=att_classes, ax=ax)
    ax.set_title("Confusion Matrix – attacktype1")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

    # ── gname confusion matrix (top-N + "Other") ─────────────────────────
    # Identify top-N most frequent groups in the validation set
    unique, counts = np.unique(all_grp_true, return_counts=True)
    top_idx = unique[np.argsort(-counts)[:top_n_groups]]
    top_labels = encoders["gname"].inverse_transform(top_idx)

    def _collapse(arr):
        """Replace any label not in top_idx with -1 (→ 'Other')."""
        return np.where(np.isin(arr, top_idx), arr, -1)

    grp_true_c = _collapse(all_grp_true)
    grp_pred_c = _collapse(all_grp_pred)

    label_map = {idx: name for idx, name in zip(top_idx.tolist(), top_labels.tolist())}
    label_map[-1] = "Other"
    ordered = sorted(label_map.keys(), key=lambda x: (x == -1, x))
    ordered_labels = [label_map[k] for k in ordered]

    cm_grp = confusion_matrix(grp_true_c, grp_pred_c, labels=ordered)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm_grp, annot=True, fmt="d", cmap="Greens",
                xticklabels=ordered_labels, yticklabels=ordered_labels, ax=ax)
    ax.set_title(f"Confusion Matrix – gname (top {top_n_groups} + Other)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


plot_confusion_matrices(model, val_loader, encoders, device, top_n_groups=10)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def analyze_regression_errors(model, val_loader, device):
    """Compute MAE/MSE/RMSE and plot predicted vs true casualties + residuals."""
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for X_b, y_att, y_suc, y_grp, y_cas in val_loader:
            X_b = X_b.to(device)
            _, _, _, cas_out = model(X_b)
            all_true.append(y_cas.cpu().numpy())
            all_pred.append(cas_out.cpu().numpy())

    y_true = np.concatenate(all_true, axis=0)  # (N, 2)
    y_pred = np.concatenate(all_pred, axis=0)  # (N, 2)

    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"Casualties Regression – MAE={mae:.4f}  MSE={mse:.4f}  RMSE={rmse:.4f}")

    # Sum over nkill + nwound for scalar comparison
    y_true_sum = y_true.sum(axis=1)
    y_pred_sum = y_pred.sum(axis=1)
    residuals  = y_true_sum - y_pred_sum

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.scatter(y_true_sum, y_pred_sum, alpha=0.3, s=5)
    lim = max(y_true_sum.max(), y_pred_sum.max())
    ax.plot([0, lim], [0, lim], "r--", label="Perfect prediction")
    ax.set_title("Predicted vs True Casualties (nkill + nwound)")
    ax.set_xlabel("True"); ax.set_ylabel("Predicted")
    ax.legend(); ax.grid(True)

    ax = axes[1]
    ax.hist(residuals, bins=50, edgecolor="black")
    ax.set_title("Residual Distribution (True – Predicted)")
    ax.set_xlabel("Residual"); ax.set_ylabel("Count")
    ax.grid(True)

    plt.tight_layout()
    plt.show()
    return {"mae": mae, "mse": mse, "rmse": rmse}


regression_metrics = analyze_regression_errors(model, val_loader, device)


In [ ]:
import joblib

# ── Save model state dict ─────────────────────────────────────────────────
torch.save(model.state_dict(), "gtd_model.pth")

# ── Save numeric scaler ───────────────────────────────────────────────────
joblib.dump(scaler, "gtd_scaler.joblib")

# ── Save all label encoders ───────────────────────────────────────────────
joblib.dump(encoders, "gtd_encoders.joblib")

print("Saved: gtd_model.pth, gtd_scaler.joblib, gtd_encoders.joblib")


In [ ]:
def predict_event(raw_event_dict, model=model, encoders=encoders, scaler=scaler,
                  device=device, binary_threshold=0.0):
    """Run inference on a single raw GTD-style event dictionary.

    Parameters
    ----------
    raw_event_dict : dict
        Dictionary with raw GTD field values, e.g.::

            {
                "iyear": 2016, "imonth": 6, "iday": 12,
                "country": "Iraq",
                "region": "Middle East & North Africa",
                "latitude": 33.3, "longitude": 44.4,
            }

    Returns
    -------
    dict
        predicted_attack_type : str
        predicted_success     : bool
        predicted_group       : str
        predicted_casualties  : float  (sum of predicted nkill + nwound)
    """
    _num_cols     = ["iyear", "imonth", "iday", "latitude", "longitude", "nkill", "nwound"]
    _cat_inf_cols = ["country", "region"]  # only these appear in the feature vector
    # Feature column order must match X_train (df.drop(columns=targets))
    _feature_cols = ["iyear", "imonth", "iday", "country", "region", "latitude", "longitude"]

    row = dict(raw_event_dict)

    # Apply label encoders for categorical feature columns
    for col in _cat_inf_cols:
        try:
            row[col] = encoders[col].transform([str(row[col])])[0]
        except ValueError:
            valid = list(encoders[col].classes_)
            raise ValueError(
                f"Unknown value '{row[col]}' for field '{col}'. "
                f"Valid options are: {valid}"
            ) from None

    # Build full num_cols array; pad nkill/nwound with 0 if absent
    num_vals = np.array([[
        float(row.get("iyear",     0)),
        float(row.get("imonth",    0)),
        float(row.get("iday",      0)),
        float(row.get("latitude",  0)),
        float(row.get("longitude", 0)),
        float(row.get("nkill",     0)),
        float(row.get("nwound",    0)),
    ]])
    num_scaled = scaler.transform(num_vals)[0]  # 7 scaled values

    # Assemble feature vector in training column order
    # _num_cols order: iyear(0) imonth(1) iday(2) lat(3) lon(4) nkill(5) nwound(6)
    # _feature_cols:   iyear    imonth    iday    country region  lat      lon
    feat = np.array([[
        num_scaled[0],          # iyear (scaled)
        num_scaled[1],          # imonth (scaled)
        num_scaled[2],          # iday (scaled)
        float(row["country"]),  # country (label-encoded)
        float(row["region"]),   # region (label-encoded)
        num_scaled[3],          # latitude (scaled)
        num_scaled[4],          # longitude (scaled)
    ]], dtype=np.float32)

    x = torch.tensor(feat, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        att_out, suc_out, grp_out, cas_out = model(x)

    att_label = encoders["attacktype1"].inverse_transform([att_out.argmax(1).item()])[0]
    suc_label = bool(suc_out.item() > binary_threshold)
    grp_label = encoders["gname"].inverse_transform([grp_out.argmax(1).item()])[0]
    cas_value = float(cas_out.cpu().numpy().sum())

    return {
        "predicted_attack_type": att_label,
        "predicted_success":     suc_label,
        "predicted_group":       grp_label,
        "predicted_casualties":  cas_value,
    }


# Example usage
sample_event = {
    "iyear": 2016, "imonth": 6, "iday": 12,
    "country": "Iraq",
    "region": "Middle East & North Africa",
    "latitude": 33.3,
    "longitude": 44.4,
}
print("Sample prediction:", predict_event(sample_event))


In [ ]:
print(df.dtypes)


iyear          float64
imonth         float64
iday           float64
country          int64
region           int64
latitude       float64
longitude      float64
nkill          float64
nwound         float64
gname            int64
attacktype1      int64
success          int64
dtype: object
